In [3]:
import polars as pl
import pandas as pd
import random

random.seed(42)

In [2]:
df = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)

df_removed_samples = pl.read_csv(
    "data/samples_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols)
)

In [3]:
### Step 1: Define Priority Columns for Cancer Detection
# We'll check columns in order of reliability

# Priority columns to check for explicit cancer mentions
PRIORITY_COLS = ["source_name", "tissue", "phenotype", "disease", "cell_type", "tumor_type"]
# Make sure these exist in your dataframe
PRIORITY_COLS = [c for c in PRIORITY_COLS if c in df.columns]

# Cancer detection patterns (same as before)
CANCER_POS = r"(?:\bcancers?\b|\btumou?rs?\b|\bmalignan(?:t|cy)\b|\bcarcinomas?\b|\bneoplasms?\b|\bmetasta(?:s|t)es?\b|\badenocarcinomas?\b|\bsarcomas?\b|\bleukemi(?:a|as)\b|\blymphom(?:a|as)\b|\bglioblastomas?\b|\bmelanomas?\b|\boncolog(?:y|ic|ical)\b)"
CANCER_NEG = r"(?:\bnormal\b|\bhealthy\b|\bctrl\b|\badjacent normal\b|\bnon[-\s]?tumou?r(?:al)?\b|\bbenign\b|\bnon[-\s]?cancer(?:ous)?\b|\bsham\b|\bunaffected\b)"
ONCO_TRAPS = r"(?:\boncophora\b|\boncorhynchus\b|\boncotic\b|\boncomodulin\b)"


### Step 2: Check Sample Name (Title/BioSample) First

def normalize_text(col_expr):
    """Normalize text for comparison"""
    return (
        col_expr.cast(pl.Utf8).fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[_/|]", " ")
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
    )


# Check sample_name/title first
sample_name_col = "title" if "title" in df.columns else "biosample"

df = df.with_columns([
    # Check if sample name mentions cancer
    normalize_text(pl.col(sample_name_col))
    .str.contains(CANCER_POS)
    .alias("cancer_in_sample_name"),

    # Check for negative indicators in sample name
    normalize_text(pl.col(sample_name_col))
    .str.contains(CANCER_NEG)
    .alias("negative_in_sample_name"),

    # Check for false positives (onco traps)
    normalize_text(pl.col(sample_name_col))
    .str.contains(ONCO_TRAPS)
    .alias("onco_trap_in_sample_name"),
])

### Step 3: Check Priority Metadata Columns

# For each priority column, check for cancer mentions
for col in PRIORITY_COLS:
    df = df.with_columns([
        normalize_text(pl.col(col))
        .str.contains(CANCER_POS)
        .alias(f"cancer_in_{col}"),

        normalize_text(pl.col(col))
        .str.contains(CANCER_NEG)
        .alias(f"negative_in_{col}"),
    ])

### Step 4: Create Confidence Levels

# Count how many priority columns mention cancer
cancer_mention_cols = [f"cancer_in_{col}" for col in PRIORITY_COLS if f"cancer_in_{col}" in df.columns]
negative_mention_cols = [f"negative_in_{col}" for col in PRIORITY_COLS if f"negative_in_{col}" in df.columns]

df = df.with_columns([
    # Count positive cancer mentions across priority columns
    pl.sum_horizontal(cancer_mention_cols).alias("n_cancer_mentions") if cancer_mention_cols else pl.lit(0).alias(
        "n_cancer_mentions"),

    # Count negative indicators
    pl.sum_horizontal(negative_mention_cols).alias("n_negative_mentions") if negative_mention_cols else pl.lit(0).alias(
        "n_negative_mentions"),
])

### Step 5: Classify into Confidence Categories

df = df.with_columns([
    pl.when(pl.col("onco_trap_in_sample_name"))
    .then(pl.lit("uncertain_onco_trap"))

    # HIGH CONFIDENCE: Cancer in sample name AND in at least one priority column
    .when(
        pl.col("cancer_in_sample_name") &
        (pl.col("n_cancer_mentions") >= 1) &
        (pl.col("n_negative_mentions") == 0)
    )
    .then(pl.lit("confident_cancer"))

    # MEDIUM CONFIDENCE: Cancer in sample name OR in multiple priority columns
    .when(
        (pl.col("cancer_in_sample_name") & (pl.col("n_negative_mentions") == 0)) |
        (pl.col("n_cancer_mentions") >= 2)
    )
    .then(pl.lit("likely_cancer"))

    # LIKELY NON-CANCER: Negative indicators present
    .when(
        (pl.col("negative_in_sample_name")) |
        (pl.col("n_negative_mentions") >= 1)
    )
    .then(pl.lit("likely_non_cancer"))

    # UNCERTAIN: Weak or no cancer signals
    .when(pl.col("n_cancer_mentions") == 1)
    .then(pl.lit("uncertain_weak_signal"))

    # UNCERTAIN: No cancer mentions at all
    .otherwise(pl.lit("uncertain_no_signal"))
    .alias("confidence_category")
])

### Step 6: Separate into Confident and Uncertain Datasets

# 100% confident cancer samples
confident_cancer_df = df.filter(pl.col("confidence_category") == "confident_cancer")

# Likely cancer (for review)
likely_cancer_df = df.filter(pl.col("confidence_category") == "likely_cancer")

# Set aside for manual review
uncertain_df = df.filter(
    pl.col("confidence_category").str.starts_with("uncertain")
)

# Non-cancer samples
non_cancer_df = df.filter(pl.col("confidence_category") == "likely_non_cancer")

### Step 7: Create Summary Report

summary = pl.DataFrame({
    "Category": [
        "Confident Cancer",
        "Likely Cancer (Review)",
        "Uncertain - Weak Signal",
        "Uncertain - No Signal",
        "Uncertain - Onco Trap",
        "Likely Non-Cancer",
        "Total"
    ],
    "Count": [
        len(confident_cancer_df),
        len(likely_cancer_df),
        len(df.filter(pl.col("confidence_category") == "uncertain_weak_signal")),
        len(df.filter(pl.col("confidence_category") == "uncertain_no_signal")),
        len(df.filter(pl.col("confidence_category") == "uncertain_onco_trap")),
        len(non_cancer_df),
        len(df)
    ]
})

print(summary)

### Step 8: Export Results

# Export confident cancer samples
confident_cancer_df.write_csv("confident_cancer_samples.csv")

# Export uncertain samples for manual review
uncertain_df.select([
    "sample_accession", "experiment_accession", "title",
    "tissue", "disease", "source_name", "phenotype",
    "confidence_category", "n_cancer_mentions", "n_negative_mentions"
]).write_csv("uncertain_samples_for_review.csv")

# Export likely cancer for validation
likely_cancer_df.write_csv("likely_cancer_for_validation.csv")

# Export non-cancer
non_cancer_df.write_csv("classified_non_cancer.csv")

print(f"✓ Exported {len(confident_cancer_df):,} confident cancer samples")
print(f"✓ Exported {len(uncertain_df):,} uncertain samples for review")
print(f"✓ Exported {len(likely_cancer_df):,} likely cancer samples for validation")
print(f"✓ Exported {len(non_cancer_df):,} non-cancer samples")

### Step 9: Show Examples from Each Category

print("=== CONFIDENT CANCER (Sample) ===")
print(confident_cancer_df.select([
    "sample_accession", "title", "tissue", "disease",
    "n_cancer_mentions"
]).head(5))

print("\n=== UNCERTAIN - NEEDS REVIEW (Sample) ===")
print(uncertain_df.select([
    "sample_accession", "title", "tissue", "disease",
    "confidence_category"
]).head(5))

print("\n=== LIKELY NON-CANCER (Sample) ===")
print(non_cancer_df.select([
    "sample_accession", "title", "tissue", "disease"
]).head(5))


shape: (7, 2)
┌─────────────────────────┬────────┐
│ Category                ┆ Count  │
│ ---                     ┆ ---    │
│ str                     ┆ i64    │
╞═════════════════════════╪════════╡
│ Confident Cancer        ┆ 23680  │
│ Likely Cancer (Review)  ┆ 41755  │
│ Uncertain - Weak Signal ┆ 28537  │
│ Uncertain - No Signal   ┆ 25571  │
│ Uncertain - Onco Trap   ┆ 0      │
│ Likely Non-Cancer       ┆ 1357   │
│ Total                   ┆ 120900 │
└─────────────────────────┴────────┘
✓ Exported 23,680 confident cancer samples
✓ Exported 54,108 uncertain samples for review
✓ Exported 41,755 likely cancer samples for validation
✓ Exported 1,357 non-cancer samples
=== CONFIDENT CANCER (Sample) ===
shape: (5, 5)
┌──────────────────┬─────────────────────────────────┬────────────┬─────────┬───────────────────┐
│ sample_accession ┆ title                           ┆ tissue     ┆ disease ┆ n_cancer_mentions │
│ ---              ┆ ---                             ┆ ---        ┆ ---     ┆ --

In [4]:
non_cancer_df

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,…,tumor_section,status_tissue,age_sacrifice,cells_loaded,gp33_status,repeat,hematopoietic_malignancy,facs_marker,tgf_beta_treatment,ena-first-public,ena-last-update,microbiome,trail_status,recipient,venus,arrayexpress-organismpart,arrayexpress-phenotype,arrayexpress-sex,arrayexpress-species,cancer_in_sample_name,negative_in_sample_name,onco_trap_in_sample_name,cancer_in_source_name,negative_in_source_name,cancer_in_tissue,negative_in_tissue,cancer_in_phenotype,negative_in_phenotype,cancer_in_disease,negative_in_disease,cancer_in_cell_type,negative_in_cell_type,cancer_in_tumor_type,negative_in_tumor_type,n_cancer_mentions,n_negative_mentions,confidence_category
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,u32,u32,str
"""CMT-290-normal""","""SRX4634743""","""RNA-Seq of canine mammary matc…","""RNA sequencing was performed u…","""SRS3735908""","""CMT-290-normal""","""RNA-Seq""","""TRANSCRIPTOMIC""","""ChIP""","""PAIRED""","""Illumina HiSeq 2500""","""SRA765600""","""nan SUB4335433 nan nan SUB4335…","""institute""","""Dept. of Biomedical Systems In…","""SRP159466""",null,"""PRJNA489087""","""9615""","""Canis lupus familiaris""","""Miniature Pinscher""",null,"""female nan female female nan f…","""normal""","""Model organism or animal""","""SRR7779518""","""39929947""","""8065849294""","""3400296216""","""TRUE""","""TRUE""","""1""","""1950129786""","""2085120920""","""2078617080""","""1950777439""","""1204069""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,true,false,false,false,false,true,false,false,false,false,false,false,false,false,0,1,"""likely_non_cancer"""
"""GSM3384282""","""SRX4671171""","""GSM3384282: CMT-290-normal; Ca…",null,"""SRS3765924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2500""","""SRA771342""","""nan GEO: GSE119810 nan nan GEO…","""center""","""NCBI""","""SRP159466""","""SAMN10034319""","""PRJNA489087""","""9615""","""Canis lupus familiaris""","""Miniature Pinscher""",null,"""nan female nan nan female nan""","""mammary normal tissue""",null,"""SRR7819837""","""39929947""","""8065849294""","""3400297199""","""TRUE""","""TRUE""","""1""","""1950129786""","""2085120920""","""2078617080""","""1950777439""","""1204069""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,true,false,false,true,false,true,false,false,false,false,false,false,false,false,0,2,"""likely_non_cancer"""
"""DRX187241""","""DRX187241""","""NextSeq 500 paired end sequenc…",null,"""DRS138631""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""NextSeq 500""","""DRA009195""","""DRA009195""","""center""","""UT-AQUA""","""DRP006092""","""SAMD00192132""","""PRJDB8934""","""9615""","""Canis lupus familiaris""",null,null,null,"""nan nan nan nan nan nan""",null,"""DRR196824""","""32611758""","""4730951736""","""2261118183""","""TRUE""","""TRUE""","""1""","""1228619800""","""1138931470""","""1112825389""","""1250562265""","""12812""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,false,false,false,false,false,false,false,false,true,false,false,false,false,0,1,"""likely_non_ca

In [11]:
# confident_cancer_df['confidence category']
# confident_cancer_df.select(["sample_accession", "title", "confidence_category"]).head(10)

# confident_cancer_df.filter(pl.col("sample_accession") == "SAMPLE_ID").select([
#     "sample_accession",
#     "title",
#     "tissue",
#     "disease",
#     "cancer_in_sample_name",
#     "negative_in_sample_name",
#     "n_cancer_mentions",
#     "n_negative_mentions",
#     "confidence_category"
# ])

confident_cancer_df.select(["n_cancer_mentions", "n_negative_mentions", "confidence_category"])


n_cancer_mentions,n_negative_mentions,confidence_category
u32,u32,str
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""
…,…,…
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""
1,0,"""confident_cancer"""


In [6]:
uncertain_df

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,…,tumor_section,status_tissue,age_sacrifice,cells_loaded,gp33_status,repeat,hematopoietic_malignancy,facs_marker,tgf_beta_treatment,ena-first-public,ena-last-update,microbiome,trail_status,recipient,venus,arrayexpress-organismpart,arrayexpress-phenotype,arrayexpress-sex,arrayexpress-species,cancer_in_sample_name,negative_in_sample_name,onco_trap_in_sample_name,cancer_in_source_name,negative_in_source_name,cancer_in_tissue,negative_in_tissue,cancer_in_phenotype,negative_in_phenotype,cancer_in_disease,negative_in_disease,cancer_in_cell_type,negative_in_cell_type,cancer_in_tumor_type,negative_in_tumor_type,n_cancer_mentions,n_negative_mentions,confidence_category
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,u32,u32,str
"""Lib2""","""SRX2358837""",null,null,"""SRS1807531""","""Lib2""","""RNA-Seq""","""TRANSCRIPTOMIC""","""RANDOM""","""SINGLE""",null,"""SRA496069""","""SUB2109767""","""institute""","""AAU""","""SRP093520""",null,"""PRJNA353983""","""9915""","""Bos indicus""","""zebu""","""not collected""","""neuter""","""horn tissue""","""Model organism or animal""","""SRR5034190""","""909364""","""363882457""","""236633098""","""TRUE""","""TRUE""","""1""","""90507536""","""90635597""","""91335407""","""91268959""","""134958""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,false,false,false,false,false,false,false,true,false,false,false,false,false,1,0,"""uncertain_weak_signal"""
"""blv_1205""","""SRX15389640""","""RNS-seq of Holstein: adult blo…","""Illumina TruSeq Stranded mRNA …","""SRS13118124""","""blv_1205""","""RNA-Seq RNA-Seq RNA-Seq RNA-Se…","""TRANSCRIPTOMIC TRANSCRIPTOMIC …","""PCR PCR PCR PCR PCR PCR""","""paired PAIRED paired paired PA…","""Illumina HiSeq 3000""","""SRA1423484""","""SUB11420854""","""institute""","""National Animal Disease Center""","""SRP376248""",null,"""PRJNA839936""","""9913""","""Bos taurus""","""Holstein nan Holstein Holstein…",null,"""nan female nan nan female nan""","""Blood nan Blood Blood nan Bloo…","""Model organism or animal""","""SRR19329936""","""71427910""","""14109050671""","""5230589191""","""TRUE""","""TRUE""",null,"""3562767829""","""3447955608""","""3441250939""","""3615300980""","""41775315""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false,false,false,false,false,false,false,false,false,false,false,false,false,false,0,0,"""uncertain_no_signal"""
"""blv_1202""","""SRX15389639""","""RNS-seq of Holstein: adult blo…","""Illumina TruSeq Stranded mRNA …","""SRS13118123""","""blv_1202""","""RNA-Seq RNA-Seq RNA-Seq RNA-Se…","""TRANSCRIPTOMIC TRANSCRIPTOMIC …","""PCR PCR PCR PCR PCR PCR""","""paired PAIRED paired paired PA…","""Illumina HiSeq 3000""","""SRA1423484""","""SUB11420854""","""institute""","""National Animal Disease Center""","""SRP376248""",null,"""PRJNA839936""","""9913""","""Bos taurus""","""Holstein nan Holstein Holstein…",null,"""nan female nan nan female nan""","""Blood nan Blood Blood nan Bloo…","""Model organism or animal""","""SRR19329937""","""63971234""","""12709653171""","""4773896224""","""TRUE""","""TRUE""",null,"""3208670671""","""3122425871""","""3119024216""","""3252796898""","""6735515""",

## Test Again on my Dataset

In [9]:
full_dataset = pl.read_csv(
    "data/combined_metadata_noncancer_removed.csv",
    schema_overrides={"group": pl.Utf8},   # <- dict of {col_name: dtype}
    infer_schema_length=0,                  # scan entire file for other cols
)

df = pl.read_csv("data/train_test.csv")

In [18]:
df = df[["experiment_accession", "bioproject", "biosample", "sample_accession", "is_cancer"]]

idx = full_dataset.columns.index("cancer_type")

full_dataset = full_dataset[full_dataset.columns[:idx+1]]

In [19]:
df.head()

# full_dataset.head()

experiment_accession,bioproject,biosample,sample_accession,is_cancer
str,str,str,str,i64
"""SRX10489019""","""PRJNA718814""","""GSM5220969""","""SRS8615268""",1
"""ERX1283444""","""PRJEB6455""","""SAMEA3724868""","""ERS1032017""",0
"""SRX17708205""","""PRJNA884362""","""GSM6601021""","""SRS15238722""",1
"""SRX4518392""","""PRJNA484951""","""GSM3323504""","""SRS3636903""",0
"""SRX9364181""","""PRJNA671877""","""GSM4860401""","""SRS7586602""",0


In [20]:
joined = full_dataset.join(
    df,
    on=["experiment_accession", "bioproject", "biosample", "sample_accession"],  # matching keys
    how="inner"  # keep only matching rows
)

In [21]:
joined

experiment_alias,experiment_accession,title,study_name,sample_accession,library_name,library_strategy,library_source,library_selection,library_layout,platform_instrument_model,submitter_accession,submitter_id,organization_type,organization_name,study_accession,biosample,bioproject,organism_tax_id,organism_scientific_name,breed,dev_stage,sex,tissue,biosamplemodel,run_accession,run_total_spots,run_total_bases,run_size,run_load_done,run_is_public,run_has_tax_analysis,a_count,c_count,g_count,t_count,n_count,cancer_type,is_cancer
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64
"""ena-EXPERIMENT-FUNCTIONAL GENO…","""ERX3526057""","""Illumina HiSeq 2500 paired end…",null,"""ERS3719163""","""2_Normal""","""RNA-Seq""","""TRANSCRIPTOMIC""","""other""","""PAIRED""","""Illumina HiSeq 2500""","""ERA2111341""","""2_Normal ena-SUBMISSION-FUNCTI…","""center""","""FUNCTIONAL GENOMICS CENTER ZUR…","""ERP117109""","""SAMEA5930682""","""PRJEB34234""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""",null,null,"""ERR3504843""","""13198053""","""3985812006""","""2403610007""","""TRUE""","""TRUE""","""1""","""968718068""","""1023813547""","""1112291056""","""878463736""","""2525599""","""carcinoma""",0
"""GSM4860400""","""SRX9364180""","""GSM4860400: Fresh-frozen canin…",null,"""SRS7586601""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860400""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""Hovawart""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899130""","""32802407""","""3292101545""","""1101440073""","""TRUE""","""TRUE""","""1""","""798070483""","""761998966""","""845415170""","""879829853""","""6787073""","""tumor*""",0
"""GSM4860401""","""SRX9364181""","""GSM4860401: Fresh-frozen canin…",null,"""SRS7586602""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina NovaSeq 6000""","""SRA1147864""","""nan GEO: GSE160124 nan nan GEO…","""center""","""NCBI""","""SRP288597""","""GSM4860401""","""PRJNA671877""","""9615""","""Canis lupus familiaris""","""French Bulldog""",null,"""nan nan nan nan nan nan""","""nan control nan nan control na…",null,"""SRR12899131""","""36359099""","""3612160297""","""1220312604""","""TRUE""","""TRUE""","""1""","""889453430""","""826675433""","""915664165""","""965546260""","""14821009""","""tumor*""",0
"""GSM3005111""","""SRX13999782""","""GSM3005111: D_Normal_Mucosa; C…",null,"""SRS11838164""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 4000""","""SRA1364912""","""GEO: GSE110661""","""center""","""NCBI""","""SRP357615""","""SAMN08550188""","""PRJNA434265""","""9615""","""Canis lupus familiaris""",null,null,"""nan nan nan nan nan nan""","""Bladder""",null,"""SRR17838664""","""9096025""","""2461885361""","""1015954149""","""TRUE""","""TRUE""",null,"""695834791""","""506960216""","""511045330""","""704969084""","""2648204""","""bladder cancer""",0
"""GSM2498357""","""SRX2581899""","""GSM2498357: CHAD-P9; Canis lup…",null,"""SRS1995924""",null,"""RNA-Seq""","""TRANSCRIPTOMIC""","""cDNA""","""PAIRED""","""Illumina HiSeq 2000""","""SRA539403""","""nan GEO: GSE95183 nan nan GEO:…","""center""","""NCBI""","""SRP100514""","""GSM2498357""","""PRJNA376380""","""9615""","""Canis lupus familiaris""","""Portuguese Water Dog""",null,"""nan nan nan nan nan nan""","""nan spleen nan nan spleen nan""",null,"""SRR5278015""","""22714044""","""2271404400""","""1615616886""","""TRUE""","""TRUE""","""1""","""578763146""","""558428200""","""547283111""","""583938897""","""2991046""","""tumor*""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""GSM3187876""","""SRX4200334""","""GSM3187876: Rh30: RH30_ENT; Mu…",null,"""S